# Sentiment Attribution: Vanilla vs ABSA with Explainability

Questo notebook confronta due approcci di sentiment analysis:
- **Vanilla**: XLM-RoBERTa sentence-level (senza awareness del target)
- **ABSA**: DeBERTa-v3 aspect-based (awareness del dirigente citato)

Usiamo **SHAP** e **Integrated Gradients** per spiegare le attribuzioni di sentiment su token-level.

In [2]:
import sys
from pathlib import Path

# Setup path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch.nn.functional as F

# Check device
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

# Load models
print("Loading Vanilla (XLM-RoBERTa)...")
vanilla_model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
vanilla_tokenizer = AutoTokenizer.from_pretrained(vanilla_model_name)
vanilla_model = AutoModelForSequenceClassification.from_pretrained(vanilla_model_name).to(device).eval()

print("Loading ABSA (DeBERTa-v3)...")
# For demo, we'll use a generic DeBERTa. In production, you'd use a fine-tuned ABSA model
absa_model_name = "microsoft/deberta-v3-base"
absa_tokenizer = AutoTokenizer.from_pretrained(absa_model_name)
absa_model = AutoModelForSequenceClassification.from_pretrained(absa_model_name).to(device).eval()

print("✓ Models loaded successfully")

## Section 1: Sample Sentences

Prepariamo frasi che evidenziano le differenze tra Vanilla e ABSA:

In [ ]:
# Sample sentences with different aspects and sentiments
samples = [
    {
        "text": "Maldini was incredible, but Cardinale's decisions have been terrible.",
        "category": "comparative_multi_target",
        "lang": "en",
    },
    {
        "text": "Without Maldini the team is much worse. His absence is really felt.",
        "category": "victim_nostalgia",
        "lang": "en",
    },
    {
        "text": "Furlani ha fatto un ottimo lavoro a differenza di Massara che ha fallito completamente.",
        "category": "comparative",
        "lang": "it",
    },
    {
        "text": "Ibra is a legend. Brought hope back to the project.",
        "category": "positive",
        "lang": "en",
    },
    {
        "text": "Cardinale's RedBird investment didn't save us from this disaster.",
        "category": "mixed",
        "lang": "en",
    },
    {
        "text": "Mancanza di un vero direttore tecnico come Maldini, ora solo delusioni.",
        "category": "nostalgia_negative",
        "lang": "it",
    },
]

df_samples = pd.DataFrame(samples)
print(f"Sample sentences loaded: {len(df_samples)}")
df_samples

## Section 2: Predict with Both Models

In [ ]:
def predict_sentiment(texts, tokenizer, model, device, model_name="model"):
    """Run inference on both models and return predictions."""
    results = []
    
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
        
        label_idx = np.argmax(probs)
        labels = ["negative", "neutral", "positive"]
        label = labels[label_idx]
        score = probs[2] - probs[0]  # pos - neg
        
        results.append({
            "text": text,
            "model": model_name,
            "label": label,
            "score": score,
            "p_neg": probs[0],
            "p_neu": probs[1],
            "p_pos": probs[2],
            "logits": logits.cpu().numpy()[0],
        })
    
    return results

# Run predictions on both models
vanilla_preds = predict_sentiment(
    df_samples["text"].tolist(), 
    vanilla_tokenizer, vanilla_model, device, "Vanilla (XLM-RoBERTa)"
)

absa_preds = predict_sentiment(
    df_samples["text"].tolist(), 
    absa_tokenizer, absa_model, device, "ABSA (DeBERTa-v3)"
)

# Combine results
results_df = pd.DataFrame(vanilla_preds + absa_preds)
print("\n=== Prediction Results ===\n")

for i, text in enumerate(df_samples["text"]):
    van = vanilla_preds[i]
    absa = absa_preds[i]
    delta = absa["score"] - van["score"]
    
    print(f"Text {i+1}: {text[:60]}...")
    print(f"  Vanilla: {van['label']:>8} (score: {van['score']:+.3f})")
    print(f"  ABSA:    {absa['label']:>8} (score: {absa['score']:+.3f})  [Δ = {delta:+.3f}]")
    print()

## Section 3: Token-Level Attribution with Integrated Gradients

In [ ]:
def integrated_gradients_attribution(text, tokenizer, model, device, target_class=2, n_steps=50):
    """
    Compute Integrated Gradients for token attribution.
    target_class: 0=negative, 1=neutral, 2=positive
    """
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    
    # Baseline: all zeros (or padding token)
    baseline_ids = torch.zeros_like(input_ids)
    
    # Forward pass with gradient tracking
    input_ids.requires_grad_(True)
    
    attributions = torch.zeros_like(input_ids, dtype=torch.float32)
    
    # Integrate gradients from baseline to input
    for step in range(n_steps):
        alpha = step / float(n_steps)
        
        # Interpolated input
        interp_ids = (1.0 - alpha) * baseline_ids + alpha * input_ids.detach()
        interp_ids.requires_grad_(True)
        
        # Forward pass
        outputs = model(input_ids=interp_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # Get target class score and compute gradients
        target_score = logits[0, target_class]
        
        if interp_ids.grad is not None:
            interp_ids.grad.zero_()
        
        target_score.backward(retain_graph=True)
        
        # Accumulate gradients
        if interp_ids.grad is not None:
            attributions += interp_ids.grad.detach().data
    
    # Average attributions
    attributions = attributions / n_steps
    
    # Sum over embedding dimension to get token attributions
    token_attributions = attributions[0].abs().sum(dim=-1).cpu().numpy()
    
    # Normalize
    max_attr = np.max(token_attributions)
    if max_attr > 0:
        token_attributions = token_attributions / max_attr
    
    return tokens, token_attributions

# Test on first sample
sample_text = df_samples.iloc[0]["text"]
print(f"Computing IG for: {sample_text}\n")

van_tokens, van_attr = integrated_gradients_attribution(
    sample_text, vanilla_tokenizer, vanilla_model, device, target_class=2, n_steps=30
)

print("Vanilla (XLM-RoBERTa) - Token Attributions (top):")
top_indices = np.argsort(van_attr)[-10:]
for idx in top_indices[::-1]:
    if idx < len(van_tokens):
        print(f"  {van_tokens[idx]:>12} : {van_attr[idx]:.3f}")


## Section 4: SHAP-like Attribution with Gradient × Input

In [ ]:
def gradient_input_attribution(text, tokenizer, model, device, target_class=2):
    """
    Simple Gradient × Input attribution (approximation of SHAP).
    """
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    
    # Enable gradients on input embeddings
    embeddings = model.get_input_embeddings()
    embed_forward = embeddings(input_ids)
    embed_forward.requires_grad_(True)
    
    # Forward pass through model
    outputs = model(inputs_embeds=embed_forward, attention_mask=attention_mask)
    logits = outputs.logits
    target_score = logits[0, target_class]
    
    # Compute gradients
    target_score.backward(retain_graph=True)
    
    # Attribution = gradient × input
    grad_embeds = embed_forward.grad
    attribution = (grad_embeds * embed_forward).sum(dim=-1)[0].cpu().detach().numpy()
    
    # Normalize to [0, 1]
    max_attr = np.abs(attribution).max()
    if max_attr > 0:
        attribution = attribution / max_attr
    
    return tokens, attribution

# Test SHAP-like attribution
van_tokens, van_shap_attr = gradient_input_attribution(
    sample_text, vanilla_tokenizer, vanilla_model, device, target_class=2
)

absa_tokens, absa_shap_attr = gradient_input_attribution(
    sample_text, absa_tokenizer, absa_model, device, target_class=2
)

print("\nGradient × Input Attribution (SHAP-like):\n")
print("Vanilla (XLM-RoBERTa):")
top_v_idx = np.argsort(np.abs(van_shap_attr))[-8:]
for idx in top_v_idx[::-1]:
    if idx < len(van_tokens):
        print(f"  {van_tokens[idx]:>12} : {van_shap_attr[idx]:+.3f}")

print("\nABSA (DeBERTa-v3):")
top_a_idx = np.argsort(np.abs(absa_shap_attr))[-8:]
for idx in top_a_idx[::-1]:
    if idx < len(absa_tokens):
        print(f"  {absa_tokens[idx]:>12} : {absa_shap_attr[idx]:+.3f}")

## Section 5: Visual Comparison of Attributions

In [ ]:
def visualize_attribution_comparison(text, van_tokens, van_attr, absa_tokens, absa_attr, figsize=(14, 5)):
    """Visualize side-by-side token attributions."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Vanilla
    ax = axes[0]
    colors_v = ['green' if x > 0 else 'red' for x in van_attr]
    ax.barh(range(len(van_tokens)), van_attr, color=colors_v, alpha=0.7)
    ax.set_yticks(range(len(van_tokens)))
    ax.set_yticklabels(van_tokens, fontsize=9)
    ax.set_xlabel("Attribution")
    ax.set_title("Vanilla (XLM-RoBERTa)\nGradient × Input Attribution")
    ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(axis='x', alpha=0.3)
    
    # ABSA
    ax = axes[1]
    colors_a = ['green' if x > 0 else 'red' for x in absa_attr]
    ax.barh(range(len(absa_tokens)), absa_attr, color=colors_a, alpha=0.7)
    ax.set_yticks(range(len(absa_tokens)))
    ax.set_yticklabels(absa_tokens, fontsize=9)
    ax.set_xlabel("Attribution")
    ax.set_title("ABSA (DeBERTa-v3)\nGradient × Input Attribution")
    ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(axis='x', alpha=0.3)
    
    fig.suptitle(f"Text: {text[:80]}...", fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Visualize for first sample
visualize_attribution_comparison(
    sample_text, van_tokens, van_shap_attr, absa_tokens, absa_shap_attr
)

## Section 6: Batch Analysis - All Samples

In [ ]:
# Compute attributions for all samples
all_results = []

for i, text in enumerate(tqdm(df_samples["text"], desc="Computing attributions")):
    van_tokens, van_attr = gradient_input_attribution(
        text, vanilla_tokenizer, vanilla_model, device, target_class=2
    )
    absa_tokens, absa_attr = gradient_input_attribution(
        text, absa_tokenizer, absa_model, device, target_class=2
    )
    
    # Get predictions
    van_pred = vanilla_preds[i]
    absa_pred = absa_preds[i]
    
    all_results.append({
        "sample_id": i,
        "text": text,
        "category": df_samples.iloc[i]["category"],
        "vanilla_label": van_pred["label"],
        "vanilla_score": van_pred["score"],
        "absa_label": absa_pred["label"],
        "absa_score": absa_pred["score"],
        "delta_score": absa_pred["score"] - van_pred["score"],
        "label_agreement": van_pred["label"] == absa_pred["label"],
        "vanilla_top_token": van_tokens[np.argmax(np.abs(van_attr))],
        "vanilla_top_attr": np.max(np.abs(van_attr)),
        "absa_top_token": absa_tokens[np.argmax(np.abs(absa_attr))],
        "absa_top_attr": np.max(np.abs(absa_attr)),
    })

results_df = pd.DataFrame(all_results)
print("\n=== Summary Statistics ===\n")
print(results_df[["vanilla_label", "vanilla_score", "absa_label", "absa_score", "delta_score", "label_agreement"]].to_string())

print(f"\n\nLabel Agreement: {results_df['label_agreement'].mean():.1%}")
print(f"Mean |Δ score|: {results_df['delta_score'].abs().mean():.3f}")
print(f"Max |Δ score|: {results_df['delta_score'].abs().max():.3f}")